# MedPredict s.r.l. — Previsione della Progressione del Diabete

**Caso d'uso aziendale:** Previsione della progressione del diabete in pazienti a rischio.

**Obiettivo:** Sviluppare un modello di regressione predittiva che, utilizzando dati clinici dei pazienti, fornisca previsioni accurate sulla progressione della malattia nel tempo.

**Dataset:** Diabetes dataset (scikit-learn) — 442 pazienti, 10 variabili cliniche, target quantitativo di progressione della malattia.

---

### Variabili indipendenti
| Feature | Descrizione |
|---------|-------------|
| age | Età del paziente |
| sex | Genere |
| bmi | Indice di massa corporea |
| bp | Pressione sanguigna media |
| s1 | Colesterolo sierico totale |
| s2 | Lipoproteine a bassa densità (LDL) |
| s3 | Lipoproteine ad alta densità (HDL) |
| s4 | Rapporto colesterolo totale / HDL |
| s5 | Trigliceridi (log) |
| s6 | Livello di glicemia |

**Target:** misura quantitativa della progressione del diabete a un anno dalla raccolta dati.

# Moduli

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import (
    LinearRegression,
    Ridge,    RidgeCV,
    Lasso,    LassoCV,
    ElasticNet, ElasticNetCV
)
from sklearn.metrics import mean_squared_error, r2_score

# Metodi comuni

La funzione `print_correlation_matrix` visualizza la matrice di correlazione come una heatmap. Prende in input un DataFrame `df` e utilizza Seaborn per generare una heatmap con la scala di colore `crest`. I valori degli indici di Pearson vengono mostrati all'interno delle celle.

In [ ]:
def print_correlation_matrix(df):
    """
    Print correlation matrix as a heatmap.

    Args:
        df (DataFrame): Input DataFrame.

    Returns:
        None
    """
    plt.figure(figsize=(12, 10), dpi=100)
    sns.set_style("whitegrid")
    sns.heatmap(
        df.corr(),
        cmap="crest",
        cbar=True,
        square=True,
        yticklabels=df.columns,
        xticklabels=df.columns,
        annot=True,
        annot_kws={'size': 10},
        fmt='.2f'
    )
    plt.tight_layout()
    plt.show()

La funzione `scale_features` esegue la standardizzazione delle features utilizzando `StandardScaler`. Applica il fit sui dati di training e trasforma sia training che test, evitando data leakage. Restituisce anche lo scaler fittato, in modo da poterlo riutilizzare in produzione senza riaddestrarlo.

In [ ]:
def scale_features(X_train, X_test):
    """
    Scale features using StandardScaler.

    Args:
        X_train (array-like): Training data.
        X_test (array-like): Test data.

    Returns:
        Tuple: (X_train_scaled, X_test_scaled, scaler)
    """
    ss = StandardScaler()
    X_train_scaled = ss.fit_transform(X_train)
    X_test_scaled = ss.transform(X_test)
    return X_train_scaled, X_test_scaled, ss

La funzione `plot_predictions` visualizza i valori predetti rispetto ai valori reali tramite uno scatter plot. Una retta tratteggiata rossa indica la previsione perfetta (predetto = reale). Più i punti sono allineati sulla retta, migliore è il modello.

In [ ]:
def plot_predictions(y_test, y_pred, title=""):
    """
    Plot predicted vs actual values.

    Args:
        y_test (array-like): True values.
        y_pred (array-like): Predicted values.
        title (str): Plot title.

    Returns:
        None
    """
    plt.figure(figsize=(6, 5), dpi=100)
    plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidths=0.4)
    lim = [min(y_test.min(), y_pred.min()) - 5, max(y_test.max(), y_pred.max()) + 5]
    plt.plot(lim, lim, 'r--', lw=2, label='Previsione perfetta')
    plt.xlabel('Valori Reali')
    plt.ylabel('Valori Predetti')
    plt.title(f'Predetto vs Reale — {title}')
    plt.legend()
    plt.tight_layout()
    plt.show()

La funzione `evaluate_model` valuta le prestazioni del modello di regressione. Calcola e stampa MSE, RMSE e R² sia per i dati di training che di test, e visualizza il grafico predetto vs reale per il test set.

In [ ]:
def evaluate_model(model_name, y_train, y_test, y_pred_train, y_pred_test):
    """
    Evaluate regression model performance.

    Args:
        model_name (str): Name of the model.
        y_train, y_test (array-like): True values.
        y_pred_train, y_pred_test (array-like): Predicted values.

    Returns:
        None
    """
    for split, y_true, y_pred in [("TRAIN", y_train, y_pred_train), ("TEST", y_test, y_pred_test)]:
        mse  = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_true, y_pred)
        print(f"{model_name} | {split}")
        print(f"  MSE  : {mse:.2f}")
        print(f"  RMSE : {rmse:.2f}")
        print(f"  R²   : {r2:.4f}")
        print()

    plot_predictions(y_test, y_pred_test, model_name)

La funzione `build_model` costruisce e valuta un modello di regressione lineare. Standardizza le feature con `scale_features`, addestra il modello specificato, genera le previsioni e chiama `evaluate_model`. Restituisce il modello addestrato e un dizionario con le metriche sul test set, usato nella tabella comparativa finale.

In [ ]:
def build_model(X_train, X_test, y_train, y_test, model, model_name):
    """
    Build and evaluate a regression model.

    Args:
        X_train, X_test (array-like): Features.
        y_train, y_test (array-like): Target values.
        model: scikit-learn regression estimator.
        model_name (str): Label for output.

    Returns:
        Tuple: (trained model, metrics dict with R2 and RMSE on test set)
    """
    X_train_scaled, X_test_scaled, _ = scale_features(X_train, X_test)

    model.fit(X_train_scaled, y_train)

    y_pred_train = model.predict(X_train_scaled)
    y_pred_test  = model.predict(X_test_scaled)

    evaluate_model(model_name, y_train, y_test, y_pred_train, y_pred_test)

    metrics = {
        'R2':   round(r2_score(y_test, y_pred_test), 4),
        'RMSE': round(float(np.sqrt(mean_squared_error(y_test, y_pred_test))), 2)
    }
    return model, metrics

La funzione `plot_learning_curve` visualizza la learning curve del modello, ovvero l'andamento dell'errore (MSE) su training e validation al variare della dimensione del training set. Prende in input un modello scikit-learn e i dati X, y.

La learning curve è uno strumento fondamentale per diagnosticare il **tradeoff bias-varianza**:
- Se le curve di training e validation sono entrambe alte con gap ridotto → il modello è in **underfitting** (alto bias)
- Se la curva di training è molto più bassa di quella di validation → il modello è in **overfitting** (alta varianza)
- Se le due curve convergono a un valore basso → il modello è ben calibrato

In [ ]:
def plot_learning_curve(model, X, y, title="", cv=5):
    """
    Plot learning curve to diagnose bias-variance tradeoff.

    Args:
        model: scikit-learn estimator (already includes scaling if needed).
        X (array-like): Features.
        y (array-like): Target values.
        title (str): Plot title.
        cv (int): Number of cross-validation folds.

    Returns:
        None
    """
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        cv=cv,
        scoring='neg_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 10),
        n_jobs=-1
    )

    train_mse = -train_scores.mean(axis=1)
    val_mse   = -val_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_std   = val_scores.std(axis=1)

    plt.figure(figsize=(9, 5), dpi=100)
    plt.plot(train_sizes, train_mse, 'o-', color='steelblue', label='Training MSE')
    plt.fill_between(train_sizes,
                     train_mse - train_std,
                     train_mse + train_std,
                     alpha=0.15, color='steelblue')
    plt.plot(train_sizes, val_mse, 'o-', color='tomato', label='Validation MSE')
    plt.fill_between(train_sizes,
                     val_mse - val_std,
                     val_mse + val_std,
                     alpha=0.15, color='tomato')
    plt.xlabel('Dimensione Training Set')
    plt.ylabel('MSE')
    plt.title(f'Learning Curve — {title}')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Caricamento del Dataset

Il dataset Diabetes di scikit-learn viene caricato e convertito in un DataFrame Pandas. Viene aggiunta la colonna `target` che rappresenta la misura quantitativa della progressione del diabete a un anno dalla raccolta dei dati clinici. Mostriamo le prime righe per una prima ispezione visiva.

In [ ]:
diabetes = load_diabetes()

df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target

print(f"Dimensioni dataset: {df.shape}")
df.head()

## Statistiche descrittive

Calcoliamo le statistiche descrittive di base per comprendere la distribuzione di ciascuna variabile clinica: media, deviazione standard, valori minimi/massimi e percentili principali.

In [ ]:
df.describe().round(3)

## Controllo valori nulli

Verifichiamo l'assenza di valori nulli nel dataset. La presenza di dati mancanti richiederebbe strategie di imputation prima di procedere con l'addestramento del modello.

In [ ]:
df.isnull().sum()

# Analisi Esplorativa dei Dati (EDA)

## Distribuzione della variabile target

Analizziamo la distribuzione della variabile target tramite istogramma e boxplot. Una distribuzione approssimativamente normale è favorevole per i modelli di regressione lineare.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

axes[0].hist(df['target'], bins=30, edgecolor='black', color='steelblue')
axes[0].set_xlabel('Progressione Diabete (target)')
axes[0].set_ylabel('Frequenza')
axes[0].set_title('Distribuzione del Target')

axes[1].boxplot(df['target'], vert=True)
axes[1].set_ylabel('Progressione Diabete (target)')
axes[1].set_title('Boxplot del Target')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

print(f"Media: {df['target'].mean():.2f} | Mediana: {df['target'].median():.2f} | Std: {df['target'].std():.2f}")

## Scatter plot: Feature vs Target

Visualizziamo la relazione tra ciascuna feature clinica e il target. Le variabili con uno scatter più allungato presentano correlazione lineare più forte e saranno candidate privilegiate per il modello.

In [ ]:
feature_cols = diabetes.feature_names

fig, axes = plt.subplots(2, 5, figsize=(20, 8), dpi=100)
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].scatter(df[col], df['target'], alpha=0.5, edgecolors='k', linewidths=0.2, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Target')
    axes[i].set_title(f'{col} vs Target')

plt.suptitle('Relazione Feature-Target', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Matrice di correlazione — Dataset completo

La matrice di correlazione di Pearson mostra le relazioni lineari tra tutte le variabili. Particolare attenzione va posta all'ultima riga/colonna che mostra la correlazione di ciascuna feature con il target. Valori elevati di correlazione tra feature indicano possibile multicollinearità.

In [ ]:
print_correlation_matrix(df)

Estraiamo e ordiniamo le correlazioni con il target per identificare rapidamente le feature più predittive.

In [ ]:
corr_with_target = df.corr()['target'].drop('target').sort_values(key=abs, ascending=False)

plt.figure(figsize=(9, 5), dpi=100)
colors = ['steelblue' if v > 0 else 'tomato' for v in corr_with_target.values]
plt.bar(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Feature Clinica')
plt.ylabel('Correlazione con Target')
plt.title('Correlazione delle Feature con la Progressione del Diabete')
plt.tight_layout()
plt.show()

print(corr_with_target)

# Feature Engineering

## Analisi della multicollinearità

Dalla matrice di correlazione emergono alcune coppie di feature fortemente correlate tra loro:
- **s1 e s2** (colesterolo totale e LDL) mostrano correlazione positiva elevata
- **s3 e s4** (HDL e rapporto colesterolo/HDL) sono negativamente correlate
- **s1 e s3** sono correlate negativamente

La presenza di multicollinearità può destabilizzare i coefficienti della regressione lineare semplice. Costruiremo perciò due versioni del dataset:
- **Dataset completo** (`df`): tutte e 10 le feature
- **Dataset ridotto** (`df2`): esclude le feature più ridondanti, mantenendo quelle con correlazione più alta con il target e minore correlazione reciproca

Dal grafico di correlazione con il target, le feature meno predittive sono **s1**, **s2** e **sex**.

In particolare:
- **s1** e **s2** sono fortemente correlate tra loro (multicollinearità) e meno correlate al target rispetto a **s5** e **bmi**
- **sex** mostra la correlazione più debole con il target

Creiamo `df2` rimuovendo queste colonne come **ipotesi da testare**: i modelli con regolarizzazione (Ridge, Lasso, ElasticNet) gestiscono già la multicollinearità internamente — in particolare Lasso può azzerare i coefficienti ridondanti autonomamente. Il confronto diretto sui dati determinerà se la rimozione manuale porta effettivamente beneficio. **La scelta finale del dataset sarà guidata dalla tabella comparativa**, non da un'assunzione a priori.

In [ ]:
df2 = df.drop(['s1', 's2', 'sex'], axis=1)
print(f"Feature dataset completo:  {list(df.drop('target', axis=1).columns)}")
print(f"Feature dataset ridotto:   {list(df2.drop('target', axis=1).columns)}")

## Matrice di correlazione — Dataset ridotto

Verifichiamo che il dataset ridotto presenti minore multicollinearità tra le feature rimaste.

In [ ]:
print_correlation_matrix(df2)

## Dataset per l'identificazione del modello

Creiamo le variabili X e y per entrambe le versioni del dataset. `X` contiene le feature e `y` il target. I due dataset verranno usati per confrontare le prestazioni dei modelli con e senza le feature ridondanti.

In [ ]:
X   = df.drop('target', axis=1).values
y   = df['target'].values

X_2 = df2.drop('target', axis=1).values
y_2 = df2['target'].values

print(f"Shape dataset completo:  X={X.shape}, y={y.shape}")
print(f"Shape dataset ridotto:   X={X_2.shape}, y={y_2.shape}")

# Regression Models

Testiamo quattro varianti della regressione lineare, ciascuna con una diversa strategia di regolarizzazione:

| Modello | Regolarizzazione | Ottimizzazione alpha |
|---------|-----------------|---------------------|
| Linear Regression | Nessuna (baseline) | — |
| Ridge | L2 | `RidgeCV` (k-fold CV su griglia log-uniforme) |
| Lasso | L1 | `LassoCV` (k-fold CV su griglia log-uniforme) |
| ElasticNet | L1 + L2 | `ElasticNetCV` (CV su griglia alpha × l1_ratio) |

Per Ridge, Lasso e ElasticNet il parametro di regolarizzazione `alpha` viene selezionato automaticamente tramite cross-validation sul training set, evitando la scelta arbitraria a mano.

Per ciascun modello confrontiamo le prestazioni sul **dataset completo** e sul **dataset ridotto**. La suddivisione train/test è 80%/20% con `random_state=0`.

Inizializziamo il dizionario che raccoglierà le metriche di tutti i modelli per la tabella comparativa finale.

In [ ]:
results = {}

## Linear Regression (baseline)

La regressione lineare semplice senza regolarizzazione costituisce il punto di riferimento (baseline) per tutti i modelli successivi. Non penalizza i coefficienti, quindi è la più sensibile alla multicollinearità e al rumore.

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

_, results['Linear Regression — completo'] = build_model(
    X_train, X_test, y_train, y_test,
    LinearRegression(),
    "Linear Regression — completo"
)

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

_, results['Linear Regression — ridotto'] = build_model(
    X_train, X_test, y_train, y_test,
    LinearRegression(),
    "Linear Regression — ridotto"
)

### Conclusioni Linear Regression
La regressione lineare senza regolarizzazione mostra già un R² positivo, confermando che le feature cliniche hanno potere predittivo sulla progressione del diabete. Costituisce il termine di confronto per valutare il beneficio della regolarizzazione nei modelli successivi.

## Ridge Regression (L2)

La regressione Ridge aggiunge una penalità L2 alla funzione di costo, proporzionale alla somma dei quadrati dei coefficienti. Questo riduce la varianza del modello e gestisce la multicollinearità distribuendo il peso tra feature correlate, senza azzerare completamente nessun coefficiente.

Il parametro `alpha` viene selezionato automaticamente tramite **`RidgeCV`**, che esegue una k-fold cross-validation (k=5) su una griglia di 50 valori distribuiti logaritmicamente nell'intervallo [10⁻³, 10³].

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

ridge_full, results['Ridge — completo'] = build_model(
    X_train, X_test, y_train, y_test,
    RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5),
    "Ridge CV — completo"
)
print(f"  → Alpha ottimale selezionato da RidgeCV: {ridge_full.alpha_:.4f}")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

ridge_ridotto, results['Ridge — ridotto'] = build_model(
    X_train, X_test, y_train, y_test,
    RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5),
    "Ridge CV — ridotto"
)
print(f"  → Alpha ottimale selezionato da RidgeCV: {ridge_ridotto.alpha_:.4f}")

### Conclusioni Ridge
Con l'alpha ottimizzato da `RidgeCV`, la regolarizzazione L2 distribuisce il peso tra feature correlate riducendo la sensibilità alla multicollinearità. È interessante notare che Ridge ottiene prestazioni **migliori sul dataset completo** rispetto al ridotto: a differenza di Lasso, che seleziona attivamente le feature, Ridge beneficia della presenza di s1 e s2 distribuendone il peso tra le variabili correlate. Rimuovere queste colonne prima di applicare Ridge rimuove informazione utile senza che il modello ne abbia bisogno — il contrario di quanto avviene per la regressione lineare semplice.

## Lasso Regression (L1)

La regressione Lasso aggiunge una penalità L1 proporzionale alla somma dei valori assoluti dei coefficienti. A differenza di Ridge, Lasso può portare alcuni coefficienti esattamente a zero, effettuando una selezione automatica delle feature più rilevanti.

Il parametro `alpha` viene selezionato automaticamente tramite **`LassoCV`**, che esegue una k-fold cross-validation (k=5) su una griglia log-uniforme. L'alpha ottimale determinerà quanti (se non zero) coefficienti vengono azzerati.

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

lasso_full, results['Lasso — completo'] = build_model(
    X_train, X_test, y_train, y_test,
    LassoCV(alphas=np.logspace(-4, 1, 50), cv=5, max_iter=10000),
    "Lasso CV — completo"
)
print(f"  → Alpha ottimale selezionato da LassoCV: {lasso_full.alpha_:.4f}")
print()
print("Coefficienti Lasso (dataset completo):")
for name, coef in zip(diabetes.feature_names, lasso_full.coef_):
    tag = '  ← azzerato' if coef == 0 else ''
    print(f"  {name:>5}: {coef:8.4f}{tag}")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

lasso_ridotto, results['Lasso — ridotto'] = build_model(
    X_train, X_test, y_train, y_test,
    LassoCV(alphas=np.logspace(-4, 1, 50), cv=5, max_iter=10000),
    "Lasso CV — ridotto"
)
print(f"  → Alpha ottimale selezionato da LassoCV: {lasso_ridotto.alpha_:.4f}")
print()
print("Coefficienti Lasso (dataset ridotto):")
for name, coef in zip(df2.drop('target', axis=1).columns, lasso_ridotto.coef_):
    tag = '  ← azzerato' if coef == 0 else ''
    print(f"  {name:>5}: {coef:8.4f}{tag}")

### Conclusioni Lasso
Con l'alpha ottimizzato da `LassoCV`, Lasso effettua una **selezione automatica delle feature**: sul dataset completo azzera già s2 (il componente con multicollinearità elevata), rendendo parzialmente ridondante la sua rimozione manuale. I coefficienti non azzerati rivelano il peso reale di ciascuna feature nella previsione della progressione del diabete. Sul dataset ridotto, con un alpha più alto, Lasso azzera anche alcune feature aggiuntive (age, s4, s6) — s4, nonostante una correlazione di 0.43 con il target, viene eliminato perché ridondante rispetto a s5 (r=0.62 tra le due).

## ElasticNet (L1 + L2)

ElasticNet combina le penalità L1 (Lasso) e L2 (Ridge) tramite il parametro `l1_ratio`. Entrambi i parametri — `alpha` e `l1_ratio` — vengono ottimizzati automaticamente tramite **`ElasticNetCV`**, che esegue una k-fold cross-validation (k=5) su una griglia bidimensionale di valori.

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

en_full, results['ElasticNet — completo'] = build_model(
    X_train, X_test, y_train, y_test,
    ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
                 alphas=np.logspace(-4, 1, 50),
                 cv=5, max_iter=10000),
    "ElasticNet CV — completo"
)
print(f"  → Alpha ottimale: {en_full.alpha_:.4f}  |  l1_ratio ottimale: {en_full.l1_ratio_:.2f}")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

en_ridotto, results['ElasticNet — ridotto'] = build_model(
    X_train, X_test, y_train, y_test,
    ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
                 alphas=np.logspace(-4, 1, 50),
                 cv=5, max_iter=10000),
    "ElasticNet CV — ridotto"
)
print(f"  → Alpha ottimale: {en_ridotto.alpha_:.4f}  |  l1_ratio ottimale: {en_ridotto.l1_ratio_:.2f}")

### Conclusioni ElasticNet
L'`l1_ratio` ottimale selezionato da `ElasticNetCV` è **1.0 su entrambi i dataset**, il che significa che la componente L2 (Ridge) non contribuisce affatto: ElasticNet degenera in puro Lasso. I risultati sono identici a quelli di Lasso (stessi MSE, RMSE, R²), confermando che la penalità L2 non aggiunge valore su questo dataset una volta ottimizzato l'alpha L1. Questa convergenza è un segnale importante: **il profilo ottimale di regolarizzazione per questo problema è L1 puro**, e Lasso è quindi il modello naturale da scegliere come finale.

# Conclusioni Finali e Scelta del Modello

## Tabella comparativa — tutti i modelli

Prima di scegliere il modello finale, raccogliamo i risultati di tutte le 8 configurazioni testate e li confrontiamo oggettivamente tramite R² e RMSE sul test set. Il modello migliore è quello con **R² più alto** (più vicino a 1) e **RMSE più basso**.

In [ ]:
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Modello', 'R2', 'RMSE']
results_df = results_df.sort_values('R2', ascending=False).reset_index(drop=True)

print("Confronto metriche sul TEST SET (ordinate per R² decrescente):")
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

colors_bar = ['steelblue' if '— ridotto' in m else 'lightsteelblue' for m in results_df['Modello']]

axes[0].barh(results_df['Modello'], results_df['R2'], color=colors_bar, edgecolor='black')
axes[0].set_xlabel('R² (Test Set)')
axes[0].set_title('R² — Confronto Modelli')
axes[0].axvline(x=results_df['R2'].max(), color='red', linestyle='--', linewidth=0.8)

axes[1].barh(results_df['Modello'], results_df['RMSE'], color=colors_bar, edgecolor='black')
axes[1].set_xlabel('RMSE (Test Set)')
axes[1].set_title('RMSE — Confronto Modelli')
axes[1].axvline(x=results_df['RMSE'].min(), color='red', linestyle='--', linewidth=0.8)

plt.suptitle('Barre scure = dataset ridotto  |  Barre chiare = dataset completo', fontsize=10)
plt.tight_layout()
plt.show()

best = results_df.iloc[0]
print(f"\nModello con R² migliore: {best['Modello']}")
print(f"  R² = {best['R2']}  |  RMSE = {best['RMSE']}")

## Dataset finale

Il dataset finale viene scelto sulla base dei risultati della tabella comparativa: preleviamo automaticamente la configurazione con il miglior R² e usiamo il dataset corrispondente (completo o ridotto). I modelli regolarizzati gestiscono la multicollinearità internamente — Lasso in particolare azzera già s2 sul dataset completo — quindi la scelta ottimale emerge dai dati, non da un'assunzione a priori.

In [ ]:
# Scelta data-driven: il dataset migliore emerge dalla tabella comparativa
best_model_name = results_df.iloc[0]['Modello']
use_ridotto = '— ridotto' in best_model_name
dataset_final = df2.copy() if use_ridotto else df.copy()
dataset_label = 'ridotto (df2)' if use_ridotto else 'completo (df)'

print(f"Modello con R² migliore: {best_model_name}")
print(f"Dataset selezionato:     {dataset_label}")
print(f"Feature del modello finale: {list(dataset_final.drop('target', axis=1).columns)}")
print(f"Shape: {dataset_final.shape}")
dataset_final.head()

## Modello finale

Sulla base della tabella comparativa selezioniamo **Lasso** come modello finale: ElasticNet ha convergito a `l1_ratio=1.0` su entrambi i dataset (puro L1), confermando che la regolarizzazione ottimale per questo problema è L1. Riaddestriamo `LassoCV` sul dataset selezionato per trovare l'alpha ottimale sul training set finale, conservando lo **scaler** fittato — necessario per applicare lo stesso pre-processing in produzione senza data leakage.

In [ ]:
X_final = dataset_final.drop('target', axis=1).values
y_final = dataset_final['target'].values

print(f"Shape X: {X_final.shape}")
print(f"Shape y: {y_final.shape}")

In [ ]:
X_final = dataset_final.drop('target', axis=1).values
y_final = dataset_final['target'].values
feature_names_final = list(dataset_final.drop('target', axis=1).columns)

print(f"Shape X: {X_final.shape}")
print(f"Shape y: {y_final.shape}")

X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=0)

# Scaling esplicito: conserviamo lo scaler per l'export
X_train_scaled, X_test_scaled, scaler_export = scale_features(X_train, X_test)

# Alpha ottimizzato tramite cross-validation sul training set finale
lasso_cv_final = LassoCV(alphas=np.logspace(-4, 1, 50), cv=5, max_iter=10000)
lasso_cv_final.fit(X_train_scaled, y_train)
optimal_alpha = lasso_cv_final.alpha_
print(f"\nAlpha ottimale per il modello finale: {optimal_alpha:.4f}")

final_model = Lasso(alpha=optimal_alpha, max_iter=10000)
final_model.fit(X_train_scaled, y_train)

y_pred_train = final_model.predict(X_train_scaled)
y_pred_test  = final_model.predict(X_test_scaled)

evaluate_model("Lasso — Modello Finale", y_train, y_test, y_pred_train, y_pred_test)

print("\nCoefficienti Lasso — Modello Finale:")
for name, coef in zip(feature_names_final, final_model.coef_):
    tag = '  ← azzerato' if coef == 0 else ''
    print(f"  {name:>5}: {coef:8.4f}{tag}")

## Cross-Validation

Eseguiamo una **k-fold cross-validation** (k=5) sull'intero dataset selezionato. Rispetto alla singola suddivisione hold-out, la cross-validation fornisce una stima più robusta della generalizzazione perché ogni osservazione viene usata sia per il training che per la validation, riducendo la dipendenza dalla casualità della singola divisione. Il gap tra R² medio in CV e R² hold-out indica quanto il singolo split possa essere ottimistico o pessimistico.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=optimal_alpha, max_iter=10000))
])

cv_scores = cross_val_score(pipeline, X_final, y_final, cv=5, scoring='r2')

print(f"R² per fold: {np.round(cv_scores, 4)}")
print(f"R² medio:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"R² hold-out: {r2_score(y_test, y_pred_test):.4f}")
print(f"Δ (CV − hold-out): {cv_scores.mean() - r2_score(y_test, y_pred_test):+.4f}")

plt.figure(figsize=(8, 4), dpi=100)
plt.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='black')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Media R²={cv_scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('R²')
plt.title('K-Fold Cross-Validation R² — Lasso Modello Finale')
plt.legend()
plt.tight_layout()
plt.show()

## Learning Curve — Analisi del Tradeoff Bias-Varianza

La learning curve mostra come variano l'errore di training e di validation al crescere della dimensione del training set. È lo strumento principale per diagnosticare il tradeoff bias-varianza:

- **Alto bias (underfitting):** entrambe le curve convergono a un valore di errore alto — il modello non ha sufficiente capacità per catturare il pattern nei dati
- **Alta varianza (overfitting):** la curva di training è molto più bassa di quella di validation — il modello si adatta troppo ai dati di training e generalizza male
- **Modello ben calibrato:** le due curve convergono a un valore di errore basso con gap contenuto

In [ ]:
plot_learning_curve(pipeline, X_final, y_final, title=f"Lasso — {dataset_label}", cv=5)

Confrontiamo la learning curve di Lasso con quella della regressione lineare semplice (baseline) per visualizzare l'effetto della regolarizzazione L1 sul tradeoff bias-varianza.

In [ ]:
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=100)

for ax, pipe, title in [
    (axes[0], pipeline_lr, "Linear Regression (baseline)"),
    (axes[1], pipeline,    f"Lasso (L1 — alpha={optimal_alpha:.4f})")
]:
    train_sizes, train_scores, val_scores = learning_curve(
        pipe, X_final, y_final,
        cv=5, scoring='neg_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    train_mse = -train_scores.mean(axis=1)
    val_mse   = -val_scores.mean(axis=1)

    ax.plot(train_sizes, train_mse, 'o-', color='steelblue', label='Training MSE')
    ax.plot(train_sizes, val_mse,   'o-', color='tomato',    label='Validation MSE')
    ax.set_xlabel('Dimensione Training Set')
    ax.set_ylabel('MSE')
    ax.set_title(f'Learning Curve — {title}')
    ax.legend()

plt.tight_layout()
plt.show()

## Esportazione del Modello

Il modello finale viene esportato tramite `pickle` insieme allo **scaler** utilizzato durante il training (lo stesso oggetto fittato su `X_train`, non una copia) e ai metadati necessari per la produzione. Questo garantisce che lo stesso pre-processing venga applicato in fase di inferenza senza rischi di disallineamento.

In [ ]:
model_artifact = {
    'model':        final_model,
    'scaler':       scaler_export,          # stesso scaler usato per il training
    'features':     feature_names_final,
    'model_name':   f'Lasso (alpha={optimal_alpha:.4f})',
    'optimal_alpha': optimal_alpha,
    'test_r2':      float(r2_score(y_test, y_pred_test)),
    'test_rmse':    float(np.sqrt(mean_squared_error(y_test, y_pred_test))),
}

with open('medpredict_diabetes_model.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print("Modello salvato in: medpredict_diabetes_model.pkl")
print(f"Modello:       {model_artifact['model_name']}")
print(f"Feature:       {model_artifact['features']}")
print(f"R² Test:       {model_artifact['test_r2']:.4f}")
print(f"RMSE Test:     {model_artifact['test_rmse']:.2f}")

In [ ]:
with open('medpredict_diabetes_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

y_pred_check = loaded['model'].predict(loaded['scaler'].transform(X_test))
print(f"Previsioni identiche dopo il ricaricamento: {np.allclose(y_pred_test, y_pred_check)}")
print("Modello pronto per la piattaforma sanitaria MedPredict.")

## Conclusioni Finali

Riassumendo il processo seguito:

- Il dataset Diabetes di scikit-learn è stato esplorato tramite statistiche descrittive, scatter plot e matrice di correlazione, identificando le feature più predittive (**bmi**, **s5**, **bp**) e le coppie con multicollinearità elevata (**s1/s2**, **s3/s4**).

- Sono stati costruiti e confrontati quattro modelli di regressione lineare con diversa regolarizzazione (**Linear Regression, Ridge, Lasso, ElasticNet**), ciascuno testato sia sul dataset completo che su quello ridotto. I parametri di regolarizzazione `alpha` (e `l1_ratio` per ElasticNet) sono stati ottimizzati automaticamente tramite cross-validation (`RidgeCV`, `LassoCV`, `ElasticNetCV`), evitando la scelta arbitraria a mano.

- L'analisi comparativa ha prodotto due osservazioni rilevanti: **Ridge performa meglio sul dataset completo** (beneficia della distribuzione del peso tra feature correlate), mentre **ElasticNet converge a puro Lasso** (`l1_ratio=1.0` su entrambi i dataset), confermando che la regolarizzazione L1 è quella ottimale per questo problema.

- La **tabella comparativa** ha guidato in modo oggettivo e data-driven sia la scelta del dataset finale che quella del tipo di modello: **Lasso**, il cui alpha ottimale è stato ricalcolato sul training set finale.

- La **learning curve** ha confermato il bilanciamento bias-varianza del modello Lasso selezionato: gap contenuto tra training e validation MSE, con entrambe le curve che convergono stabilmente.

- La **k-fold cross-validation** (k=5) ha prodotto un R² medio più alto rispetto all'hold-out, evidenziando come la singola suddivisione possa essere leggermente pessimistica con 442 campioni.

- Il modello è stato esportato con **pickle** — insieme allo stesso scaler usato per il training — ed è pronto per essere integrato nella piattaforma sanitaria di MedPredict.

### Alcune cifre di massima
- Il modello spiega circa il 43–48% della varianza della progressione del diabete con sole variabili cliniche di base
- L'aggiunta di dati longitudinali (storico del paziente) e di biomarcatori aggiuntivi potrebbe incrementare significativamente la capacità predittiva del modello in produzione